# Solveit CRAFT UI Architecture

> Map of every UI feature notebook in `nbs/craft/`: what it owns, why it exists, how `craft_dev.build` assembles it into `dist/`, and which Solveit DOM and runtime contracts it relies on.

## Build Pipeline & Load-Order Layers

`craft_dev.build.build_dist()` scans `nbs/craft/*.ipynb` in **sorted filename order** and concatenates exported cells into `dist/`:

- `#| export_ft <file>` evaluates Python `fastcore.xml` `FT` trees at build time and emits compact HTML/SVG strings into `window.__craftTpl` (and config tokens into `window.__craftCfg`).
- `#| export_js <file>` appends raw JavaScript cells to `<file>` with `/* %% <nb> #<cell_id> */` source markers.
- `#| export_js_iife <file> <group>` wraps multiple cells in the same notebook that share `(file, group)` into a single `(() => { ... })();` block, letting large modules be split across narrated cells without losing lexical scope.
- `#| export_css <file>` appends CSS rules to `<file>` in notebook order, so later layers naturally override earlier rules in the cascade.

Every notebook in `nbs/craft/` must have a unique numeric prefix (`build_dist` enforces this) because prefix order **is** both JS initialization order and CSS cascade order.

## Feature Map

| Prefix | Notebook | Output(s) | Globals / Responsibilities |
|---|---|---|---|
| `00` | `00_architecture.ipynb` | *(docs only)* | Architecture guide, dependency layers, and runtime contracts |
| `01` | `01_theme_and_tokens.ipynb` | `CRAFT.css` | Tokyo Night Pure palette, `--craft-prose` / `--craft-code` type scale, Encode Sans & Iosevka webfonts, modals, find/replace, `#vars-sb`, task lists, raw cells, ANSI traceback contrast |
| `02` | `02_hotkey_builtins.ipynb` | `hotkey_builtins.js` | Reference snapshot of upstream Solveit keyboard shortcuts |
| `03` | `03_hotkey_manager.ipynb` | `CRAFT.js`, `hotkey_manager.js`, `CRAFT.css` | `window.__hotkeyRegister` bootstrap queue, `window.solveitHotkeys` registry/dispatcher, `.hk-overlay` settings modal |
| `04` | `04_cm_core.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.__craftCMCore`: CodeMirror 6 loader (local bundle + `esm.sh` fallback), syntax themes, language loaders, `createInlineEditor`, shared DOM/HTMX/modal primitives (`el`, `qs`, `qsa`, `on`, `patch`, `store`, `mountTpl`, `mountModal`, `MODAL_OPEN_SEL`, `typingBlocked`, `htmxPost`, `onDomChange`, `allCells`, `selectedCells`, `primaryCell`, `cellMtype`, `dlgKey`, `sleep`, `notify`, `notifyErr`, `deleteCellEl`, `followRename`) |
| `05` | `05_dialog_chrome.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftMobile` mode toggle, auto-hiding top nav, compact Monaco line height, `Mod+Shift+P` launcher, `window.craftIdleComposer`, `window.craftA11y` |
| `06` | `06_file_tree.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.solveitFileTree`: Quercus sidebar tree, inline CRUD, copy/move/delete, uploads, resizable drawer |
| `07` | `07_cell_presentation.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftCellChains`, `window.craftActionMenus`, `window.craftOmit`, `window.craftParts`, pinned/exported/skipped cues, desktop hover/selection outlines |
| `08` | `08_tables.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftTables`: scroll wrapper, numeric right-alignment, click-to-sort flat headers, Copy TSV, light-mode overrides |
| `09` | `09_cell_insertion.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftInsertPreview`, `window.craftDesktopUX` (hover-edge drag-to-insert, fan picker, `Shift+A`/`Shift+B` seam picker, slash-morph, click-margin escape), `window.craftImageDrop` |
| `10` | `10_code_editor_c2.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftC2`: inline CM6 code editor, folding, line numbers, completions, signature help, ghost text, inline AI |
| `11` | `11_markdown_editor.ipynb` | `CRAFT.js`, `craft_md.js`, `CRAFT.css` | `craft_md.js` loader, `window.craftMd`: Obsidian-style Live Preview for note/prompt/response and plain-text raw editor |
| `12` | `12_mobile.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftMobileBar` bottom editing toolbar, edge-bulge insertion, and all `html.craft-mobile` responsive overrides |
| `13` | `13_context_control.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftContext`: transactional visibility engine, All-But one-shot scoping, Context Lenses, Saved Presets, Context Budget Meter, What the AI Sees preview, Traceback Quick-Fix |
| `14` | `14_font_settings.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftFonts`: per-cell-type font family/size/line-height/weight modal with live preview, Monaco sync, persistence, and CRAFT.css snippet export |
| `15` | `15_structural_ops.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftStruct`: multi-cell batch bar, Merge Cells, Partition Dialog (Split/Fork/Extract), Send to Another Dialog, Alt/Hyper drag & keyboard reordering, heading section actions |
| `16` | `16_navigation_reference.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftNavRef`: collapsible Minimap / Outline Rail, Sticky Reference Pane, session-scoped Stale-Output badges, Scratchpad / ephemeral cell flags |
| `17` | `17_cell_history.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftHistory`: session per-cell version snapshots, inline AI edit diff/Accept/Revert bar, and Cell History restore modal |
| `18` | `18_tts_reader.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftTTS`: Web Speech API reader with Markdown/code/table cleaning, sentence chunking, active-cell highlighting, and floating player bar |
| `19` | `19_bug_capture.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftBugCapture`: floating bug-report trigger, DOM/console/network snapshot capture, and submission modal |

## Core Solveit DOM & Lifecycle Constraints

- **HTMX + WebSocket card replacement:** Solveit frequently replaces `.editable` cards in `#dialog-container` when cells run, save, or sync. Features must be idempotent and re-attach cleanly via `window.__craftCMCore.onDomChange(...)` rather than assuming DOM nodes live forever.
- **Hot-reload safety:** Every feature IIFE calls `window.<global>?.disable?.()` at startup before registering listeners or observers, so reloading `CRAFT.js` during development never leaks duplicate handlers.
- **Page-local vs server-backed state:** Omit (`craftOmit`) and the third hidden state in `craftParts` are strictly page-local DOM state keyed by stable message IDs; they never mutate dialog JSON or AI-context inclusion.
- **Standalone runtime deployment:** End-user installs via `setup.sh` copy only the built root assets (`CRAFT.css`, `CRAFT.js`, `craft_md.js`, `hotkey_manager.js`, `hotkey_builtins.js`, `quercus-treeview.*`, `vscode-icons/`) without requiring Python, `uv`, or Node.